# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one page** (`client_hash_id` + `content_hash_id`), built from the daily table.

The warehouse stores **one row per page per day**. I turn that into **one row per page** with two time windows:

| Window | Dates | What comes from it |
|---|---|---|
| **Before** (features) | 1–28 Feb 2026 | Feb impressions, Feb average position |
| **Decision moment** | 1 Mar 2026 | everything a feature uses must be known by this date |
| **After** (outcome) | 1–31 Mar 2026 | did the page decline? |

**Outcome:** `is_declining = 1` when the page's **impressions per day** in March are more than 20% lower than in February. I use *per day* because February has 28 days and March has 31. Comparing totals would make March look ~11% better for free.

**Which pages are included:**
- only clients whose Search Console history started **on or before 1 Feb 2026**, so every page has a full February (the panel is unbalanced: clients started on different dates);
- only pages created **before 1 Feb 2026**, so they existed for all of February;
- only pages with **at least 100 impressions in February**, so one or two lost impressions can't flip the label;
- only days where `gsc_data_available IS TRUE`. The flag can be TRUE, FALSE, or empty (NULL), so `IS TRUE` is the only safe filter.

Both months are mid-panel. I don't touch the last month (June 2026); the card calls it the sealed test month.

This fixes the main limit from Week 2: there, signals and outcome came from the **same** 90 days. Here, signals come **before** the outcome.

The code connects to the warehouse and checks how many clients have a full February.

**Result (you write this after Run all):** _How many clients have a full February?_

In [ ]:
# Setup: connect DuckDB to the warehouse (token comes from the Colab Secrets panel, never typed here)
%pip -q install duckdb
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
# Only the two months I need (a list of paths; hf:// doesn't support {a,b} globs)
FACT_FEB_MAR = (f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
                f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])")

BEFORE_START, DECISION, AFTER_END = "2026-02-01", "2026-03-01", "2026-03-31"

# How many clients have a full February of Search Console history?
print(con.sql(f"""
    SELECT COUNT(*) AS all_clients,
           COUNT(*) FILTER (WHERE gsc_data_start <= DATE '{BEFORE_START}') AS full_february,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS no_gsc_start_date
    FROM {DIM_CLIENTS}
""").df().to_string(index=False))

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `content_age_days` (from `content_created_date`) | **feature** | creation date is known before 1 Mar |
| `word_count` | **feature** | page length; see the snapshot limit in Section 4 |
| `search_volume` | **feature** | demand for the page's keyword; see Section 4 |
| `feb_avg_position` (from `gsc_avg_position`, Feb only) | **feature** | measured before 1 Mar; weighted by impressions so busy days count more |
| `feb_impressions` (from `gsc_impressions`, Feb only) | **feature** | measured before 1 Mar |
| `mar_impressions` (from `gsc_impressions`, Mar only) | **label ingredient** | it *is* the after window; using it = copying the answer |
| `is_declining` | **label** | what I compare signals against |
| `client_hash_id`, `content_hash_id`, `report_date`, `month` | **context** | joining, grouping, splitting by client, picking windows |
| `gsc_data_available`, `gsc_data_start` | **context** | filters for usable rows and full history |
| `content_updated_date`, `last_optimized_date` | **excluded** | can be *after* 1 Mar, so they leak the future |
| `is_deleted`, `is_published` | **excluded** | today's status, not the status on 1 Mar |
| `provider_used`, `model_used` | **excluded** | product metadata, not page signals (data dictionary) |
| all `ga4_*`, `sessions_*`, `ai_*` columns | **excluded (for now)** | GA4 starts later for many clients and has NULL gaps; keeping five Search Console features first |

**Deliberately excluded, in one line:** anything dated after 1 March, especially March impressions, because that is the answer itself.

The code writes this table down as lists and checks that no feature is also a label or excluded field.

In [ ]:
# The contract as code: every field in exactly one bucket
FEATURES = ["content_age_days", "word_count", "search_volume", "feb_avg_position", "feb_impressions"]
LABEL = ["is_declining", "mar_impressions"]            # the answer and its ingredient
CONTEXT = ["client_hash_id", "content_hash_id", "report_date", "month",
           "gsc_data_available", "gsc_data_start"]
EXCLUDED = ["content_updated_date", "last_optimized_date", "is_deleted", "is_published",
            "provider_used", "model_used"]

buckets = {"feature": FEATURES, "label": LABEL, "context": CONTEXT, "excluded": EXCLUDED}
all_fields = [f for group in buckets.values() for f in group]
assert len(all_fields) == len(set(all_fields)), "a field sits in two buckets"
assert not set(FEATURES) & (set(LABEL) | set(EXCLUDED)), "a label or excluded field is a feature"

for name, group in buckets.items():
    print(f"{name:9} ({len(group)}): {', '.join(group)}")
print("\nCheck passed: every field is in exactly one bucket, and no feature is a label or excluded.")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three facts, three queries, all on the Feb–Mar slice:

1. **Grain:** one row in the daily table really is one page on one day (no duplicates).
2. **Counts and dates:** how many rows and pages each month has, and that the dates run 1 Feb → 31 Mar.
3. **Availability:** how many rows have usable Search Console data (`IS TRUE`), and how many are FALSE or empty.

Then I build the **five-feature frame**, and finally spring **the trap**.

In [ ]:
# Query 1 — GRAIN: is (report_date, client, content) unique? Zero rows back = yes.
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_FEB_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate page-days found: {len(dupes)}")
print("Grain holds: one row = one page on one day" if dupes.empty else "Grain BROKEN, see rows above")

In [ ]:
# Query 2 — COUNTS + DATE SPAN per month
print(con.sql(f"""
    SELECT month,
           COUNT(*)                         AS rows,
           COUNT(DISTINCT content_hash_id)  AS pages,
           COUNT(DISTINCT client_hash_id)   AS clients,
           MIN(report_date)                 AS first_day,
           MAX(report_date)                 AS last_day,
           COUNT(DISTINCT report_date)      AS days
    FROM {FACT_FEB_MAR}
    GROUP BY month
    ORDER BY month
""").df().to_string(index=False))

In [ ]:
# Query 3 — AVAILABILITY: how many rows survive the IS TRUE filter?
avail = con.sql(f"""
    SELECT month,
           COUNT(*)                                              AS rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)    AS gsc_true,
           COUNT(*) FILTER (WHERE gsc_data_available = FALSE)    AS gsc_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)    AS gsc_null
    FROM {FACT_FEB_MAR}
    GROUP BY month
    ORDER BY month
""").df()
avail["survive_%"] = (avail["gsc_true"] / avail["rows"] * 100).round(1)
print(avail.to_string(index=False))

### The five-feature frame

One SQL query does the heavy work inside DuckDB and hands back **one small row per page**. Only that small table comes into pandas.

**Available when?** (each feature must be known on 1 March):
- `content_age_days`: knowable at 1 Mar because the creation date is in the past (days from creation to 1 Mar).
- `word_count`: knowable at 1 Mar because the page already existed, but the value is today's snapshot (Section 4).
- `search_volume`: knowable at 1 Mar because it describes keyword demand, also a snapshot (Section 4).
- `feb_avg_position`: knowable at 1 Mar because it only uses 1–28 Feb.
- `feb_impressions`: knowable at 1 Mar because it only uses 1–28 Feb.

`mar_impressions` is built in the same query **only to make the label**. It is never a feature.

In [ ]:
# Build one row per page: Feb features + March outcome
frame = con.sql(f"""
    WITH ok_clients AS (
        SELECT client_hash_id FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{BEFORE_START}'
    ),
    per_page AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_impressions,
               SUM(f.gsc_avg_position * f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}')
                 / NULLIF(SUM(f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}'), 0)
                                                                                         AS feb_avg_position,
               COUNT(*) FILTER (WHERE f.report_date >= DATE '{DECISION}')                AS mar_days,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date >= DATE '{DECISION}')  AS mar_impressions
        FROM {FACT_FEB_MAR} f
        JOIN ok_clients USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.content_hash_id,
           date_diff('day', d.content_created_date, DATE '{DECISION}') AS content_age_days,
           d.word_count, d.search_volume,
           p.feb_avg_position, p.feb_impressions, p.mar_days, p.mar_impressions
    FROM per_page p
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE d.content_created_date < DATE '{BEFORE_START}'
      AND p.feb_impressions >= 100
""").df()

pages_before = len(frame)
frame = frame[frame["mar_days"] > 0].copy()          # pages with no usable March rows: label unknown
dropped_no_march = pages_before - len(frame)

# Label: March impressions per day more than 20% below February's
feb_per_day = frame["feb_impressions"] / 28
mar_per_day = frame["mar_impressions"] / 31
frame["is_declining"] = (mar_per_day < 0.8 * feb_per_day).astype(int)

print(f"Pages in the frame: {len(frame):,} from {frame['client_hash_id'].nunique()} clients")
print(f"Dropped (no usable March data): {dropped_no_march:,}")
print(f"Base rate: {frame['is_declining'].mean() * 100:.1f}% declining\n")
print("Feature summary (no rows shown):")
print(frame[FEATURES].astype(float).describe().T[["count", "mean", "50%", "min", "max"]].round(1).to_string())
print("\nMissing per feature (%):")
print((frame[FEATURES].isna().mean() * 100).round(1).to_string())

### The trap: add a label-derived column on purpose

A quick score shows the effect. I train a small model and measure **AUC** on clients it never saw (whole clients go to train or test, never both).

- **AUC 0.5** = coin flip, **AUC 1.0** = perfect.

First with the 5 honest features. Then I add `mar_impressions` (the answer's ingredient) and watch the score jump toward perfect. Then I delete it and keep the honest number. The same lesson as warm-up notebook 02, now on the real warehouse.

In [ ]:
# Quick score: honest features vs. honest + one leaked column
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, frame["is_declining"], groups=frame["client_hash_id"]))
train, test = frame.iloc[train_idx], frame.iloc[test_idx]
assert not set(train["client_hash_id"]) & set(test["client_hash_id"]), "a client is in both sides"

def quick_auc(cols):
    model = HistGradientBoostingClassifier(max_depth=3, random_state=42)
    model.fit(train[cols], train["is_declining"])
    return roc_auc_score(test["is_declining"], model.predict_proba(test[cols])[:, 1])

honest = quick_auc(FEATURES)
leaky = quick_auc(FEATURES + ["mar_impressions"])
print(f"Test: {len(test):,} pages from {test['client_hash_id'].nunique()} unseen clients "
      f"(base rate {test['is_declining'].mean() * 100:.1f}%)")
print(f"AUC, 5 honest features:          {honest:.3f}")
print(f"AUC, + mar_impressions (LEAK):   {leaky:.3f}   <- too good: it contains the answer")

# Delete the leak and keep the honest number
final = frame[["client_hash_id", "content_hash_id"] + FEATURES + ["is_declining"]]
assert "mar_impressions" not in final.columns
print(f"\nLeak removed. Final frame: {final.shape[0]:,} pages x {len(FEATURES)} features. "
      f"Honest AUC kept: {honest:.3f}")

**Result (you write this after Run all):** _Did the grain hold? How many rows and pages in Feb and Mar, and what % survived `IS TRUE`? How many pages ended up in the frame, what is the base rate, and how far did AUC jump with the leak?_

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: `dim_content` is a present-day snapshot, not a 1-March snapshot.**

`word_count` and `search_volume` come from `dim_content`, which describes each page **as it is in the release** (mid-2026), not as it was on 1 March. If a page was edited after 1 March, its word count today may not match the page that existed in February. That makes these two features slightly "from the future." The code counts how many pages in my frame were updated or optimized after the decision moment, so the size of the problem is measured, not guessed.

**Other limits:**
- **Unbalanced panel:** clients without a full February are dropped, so results describe clients with longer history.
- **Survivor pages:** pages need ≥100 Feb impressions and usable March data. Tiny or vanished pages are not represented.
- **One month pair:** Feb → Mar is one window. A pattern could differ in other months; the capstone can repeat it on other pairs.
- **Search Console only:** no GA4 engagement features yet (later start dates, NULL gaps).
- **Still observational:** a feature that goes with decline does not cause it.

**Result (you write this after Run all):** _What % of pages were updated or optimized after 1 March? Is the snapshot problem small or big?_

In [ ]:
# How big is the snapshot problem? Pages in my frame changed after 1 March
snap = con.sql(f"""
    SELECT COUNT(*) AS pages,
           COUNT(*) FILTER (WHERE content_updated_date > DATE '{DECISION}') AS updated_after_decision,
           COUNT(*) FILTER (WHERE last_optimized_date  > DATE '{DECISION}') AS optimized_after_decision
    FROM {DIM_CONTENT} d
    JOIN (SELECT client_hash_id, content_hash_id FROM final) k USING (client_hash_id, content_hash_id)
""").df()
snap["updated_%"] = (snap["updated_after_decision"] / snap["pages"] * 100).round(1)
snap["optimized_%"] = (snap["optimized_after_decision"] / snap["pages"] * 100).round(1)
print(snap.to_string(index=False))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.